# Fit a low-rank update while freezing the base

Prerequisites: Matrix factorization, gradients and parameter counts.

A low-rank adapter represents an update ΔW=BA with rank at most r while leaving a base matrix W₀ unchanged. The companion fits a rank-two target update for a 6×8 matrix. It initializes A with small random values and B at zero, then updates both using gradients of the reconstruction loss.

Initializing both factors to zero would make both gradients zero at the start, trapping this factorization. A frozen base also needs an actual immutability check, not just a label on a diagram. The notebook compares base bytes before and after fitting and reports adapter versus dense parameter counts. It is a matrix-fit demonstration, not a language-model fine-tuning benchmark or a claim that every task admits a low-rank solution.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Base | W₀ frozen | Retain pretrained parameters |
| Adapter | B(6×2) A(2×8) | Rank ≤2 |
| Training | Gradient updates to A and B | Check loss and base identity |

## Predict

Rank two uses 2×8+6×2=28 adapter parameters versus 48 dense entries. The base remains unchanged and loss should fall from its initial value.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import extensions as m

In [ ]:
result=m.lora()
assert result['base_unchanged']
assert result['adapter_parameters']==28
assert result['rank']<=2
assert result['final_loss']<result['initial_loss']
print(result)

## Perturb and explain

Fit a full-rank target with ranks one and two. Separate optimization failure from the best possible low-rank approximation error.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

Why avoid setting both adapter factors to zero?

1. It changes the frozen base
2. Both factor gradients can vanish initially
3. It always exceeds memory

Answer and explanations: 1: The base is a separate parameter. 2: Each factor gradient includes the other factor. 3: The issue here is optimization, not allocation.

## Primary evidence

- [Hu et al. — LoRA](https://arxiv.org/abs/2106.09685)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.